# Customer Churn Risk Modelling

**Author:** Khaliq Lamid

**Project Type:** Personal Data Science Project

**Dataset:** Telco Customer Churn 

---


Customer churn refers to when a subscriber cancels or stops using a service. Retaining an existing customer costs significantly less than acquiring a new one, which makes early identification of at risk customers a valuable business problem.

This project builds and evaluates a machine learning pipeline to predict churn using the Telco Customer Churn dataset, a publicly available dataset containing records for 7,043 customers across 20 features including contract type, tenure and monthly charges. The target variable is binary: churned or not churned.

The pipeline covers data cleaning, exploratory analysis, one hot encoding of categorical variables and training two classification models: Logistic Regression and Random Forest. Models are evaluated using Precision, Recall, F1-score and ROC-AUC.

A key focus is probability threshold tuning, where the default classification cut off of 0.5 is tested across a range from 0.30 to 0.50. Because missing a customer who churns is typically more costly than incorrectly flagging one who would have stayed, adjusting this threshold allows the model to better reflect real business priorities. This analysis identifies which model and threshold combination offers the best trade off between catching churners and minimising false alarms.

## Table of Contents

1. Problem Definition and Dataset Overview
2. Data Loading and Initial Inspection
3. Exploratory Data Analysis
4. Data Cleaning and Preprocessing
5. Feature Encoding and Target Preparation
6. Train/Test Split
7. Model 1: Logistic Regression
8. Model 2: Random Forest
9. Probability Threshold Tuning
10. Feature Importance Analysis
11. Model Persistence
12. Conclusion and Reflection


## 1. Problem Definition and Dataset Overview

### Objective
This project builds a supervised machine learning model to predict whether a Telco customer will cancel their service. The model uses information already held by the business such as what services a customer subscribes to, how long they have been a customer and what type of contract they are on, to identify who is likely to leave before they do.

### Dataset
The Telco Customer Churn dataset contains 7,043 customer records with 21 features covering:
- **Demographic information** — gender, senior citizen status, partner/dependents
- **Service subscriptions** — phone, internet, online security, tech support, streaming
- **Account information** — tenure, contract type, payment method, monthly charges, total charges
- **Target variable** — `Churn` (Yes/No)

### Business Framing
Acquiring a new customer costs more than keeping an existing one. A model that flags high risk customers early gives retention teams a specific list to act on, rather than marketing broadly and hoping for the best. The goal is not just accuracy; it is identifying the right customers in time to do something about it.

## 2. Data Loading and Initial Inspection


Loading the Telco Customer Churn CSV file and checking the dataset shape, columns, and a preview of the data.


In [ ]:
import pandas as pd
df = pd.read_csv("/content/Telco_Cusomer_Churn.csv")
print(df.shape)
print(df.columns)
df.head()


(7043, 21)
Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Most columns are stored as expected. The exception is TotalCharges which pandas reads as text instead of a number. This gets fixed in preprocessing before any modelling takes place.

In [ ]:
df.dtypes

,0
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


## 3. Exploratory Data Analysis

Before building any model, it's worth understanding what the data actually shows. Crosstabs and group statistics are used here to spot patterns between customer features and churn, giving the modelling decisions some grounding.


### 3.1 Churn distribution (overall class balance)


In [ ]:
counts = df["Churn"].value_counts()
print(counts)

Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [ ]:
percentages = df["Churn"].value_counts(normalize=True) * 100
print(percentages)

Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


73% of customers didn't churn only 27% did. That imbalance matters because a model could predict no churn for everyone and still hit 73% accuracy without actually learning anything useful. Precision, Recall, F1-score and ROC-AUC are used instead.

### 3.2 Churn rate by Contract type


In [ ]:
pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100

Churn,No,Yes
Contract,,
Month-to-month,57.290323,42.709677
One year,88.730482,11.269518
Two year,97.168142,2.831858


Contract type is the clearest churn signal in the data. Month to month customers churn at a noticeably higher rate than those on one or two year contracts making it one of the most useful features for the model.

### 3.3 Tenure and monthly charges across churn outcomes


In [ ]:
df.groupby("Churn")["MonthlyCharges"].describe()

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
No,5174.0,61.265124,31.092648,18.25,25.10,64.425,88.4,118.75
Yes,1869.0,74.441332,24.666053,18.85,56.15,79.650,94.2,118.35


In [ ]:
df.groupby("Churn")["tenure"].describe()


,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
No,5174.0,37.569965,24.113777,0.0,15.0,38.0,61.0,72.0
Yes,1869.0,17.979133,19.531123,1.0,2.0,10.0,29.0,72.0


Customers who churned had shorter tenures and paid slightly more per month than those who stayed. Both variables separate the two groups well enough to be worth including in the model.

### 3.4 Churn rate by Internet Service type


In [ ]:
pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100


Churn,No,Yes
InternetService,,
DSL,81.040892,18.959108
Fiber optic,58.107235,41.892765
No,92.595020,7.404980


Fibre optic customers churn at a higher rate than DSL or no internet customers. Whether that's down to price or service quality can't be determined from this data alone but the pattern is clear enough that internet service type will be a useful predictor.

### 3.5 Churn rate by Payment Method


In [ ]:
pd.crosstab(df["PaymentMethod"], df["Churn"], normalize="index") * 100


Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),83.290155,16.709845
Credit card (automatic),84.756899,15.243101
Electronic check,54.714588,45.285412
Mailed check,80.893300,19.106700


Electronic check users churn considerably more than those paying by automatic bank transfer or credit card. This likely overlaps with the month to month contract group customers with less commitment to the service tend to cluster around both.

### 3.6 Cross analysis: Payment Method vs Contract type


In [ ]:
pd.crosstab(df["PaymentMethod"], df["Contract"], normalize="index") * 100


Contract,Month-to-month,One year,Two year
PaymentMethod,,,
Bank transfer (automatic),38.147668,25.323834,36.528497
Credit card (automatic),35.676741,26.149803,38.173456
Electronic check,78.224101,14.672304,7.103594
Mailed check,55.397022,20.905707,23.697270


As expected, electronic check users skew heavily toward month to month contracts. Both variables are worth keeping in the model but they're partly measuring the same underlying behaviour that is low commitment.

### 3.7 Churn rate by Tech Support


In [ ]:
pd.crosstab(df["TechSupport"], df["Churn"], normalize="index") * 100


Churn,No,Yes
TechSupport,,
No,58.364526,41.635474
No internet service,92.595020,7.404980
Yes,84.833659,15.166341


Customers without tech support churn at a noticeably higher rate. It's hard to say whether that's because they run into unresolved issues or simply because fewer add ons mean it's easier to walk away or probably both.

## 4. Data Cleaning and Preprocessing

TotalCharges is converted from text to numeric here with any non convertible entries coerced to NaN for handling in the next step.


### 4.1 Identifying the problem in TotalCharges


Checking how many rows contain whitespace only TotalCharges values


In [ ]:
(df["TotalCharges"].str.strip() == "").sum()


np.int64(11)

### 4.2 Converting TotalCharges to numeric


Converting TotalCharges to numeric anything that can't be parsed becomes NaN.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")


In [ ]:
df["TotalCharges"].isna().sum()


np.int64(11)

### 4.3 Investigating the missing values


Checking the characteristics of rows where TotalCharges is now missing. These are likely new customers with zero tenure.


In [ ]:
df.loc[df["TotalCharges"].isna(), ["tenure", "MonthlyCharges", "Churn"]].head(15)


,tenure,MonthlyCharges,Churn
488,0,52.55,No
753,0,20.25,No
936,0,80.85,No
1082,0,25.75,No
1340,0,56.05,No
3331,0,19.85,No
3826,0,25.35,No
4380,0,20.00,No
5218,0,19.70,No
6670,0,73.35,No


Every missing TotalCharges row has a tenure of 0 and a churn of No and New customers not yet billed.

### 4.4 Filling missing values


In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)


In [ ]:
df["TotalCharges"].isna().sum()


np.int64(0)

## 5. Feature Encoding and Target Preparation

Preparing the data for modelling: encoding the target variable to numeric, separating features and target and applying one hot encoding to categorical variables.


### 5.1 Encoding the Churn target


Converting Churn from 'Yes/ No' to 1/0 for use as a binary classification target.


In [ ]:
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
df["Churn"].value_counts()


,count
Churn,
0,5174
1,1869


### 5.2 Separating target and features


Dropping the target and the customerID

In [ ]:
y = df["Churn"]

In [ ]:
X = df.drop(columns=["Churn", "customerID"])

### 5.3 One hot encoding categorical features


Converting categorical variables to numeric using one hot encoding with drop_first=True to avoid the dummy variable trap.


In [ ]:
X = pd.get_dummies(X, drop_first=True)

In [ ]:
X.shape

(7043, 30)

## 6. Train / Test Split

Splitting the dataset into 80% training and 20% testing using a fixed random state for reproducibility.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape


((5634, 30), (1409, 30))

## 7. Model 1: Logistic Regression

Logistic Regression is chosen as the baseline classification model because it is interpretable well suited to binary classification problems and provides easily comparable feature coefficients.


### 7.1 Training the model


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]


### 7.2 Evaluation metrics


Evaluating the model using the confusion matrix, classification report (Precision, Recall, F1-score) and ROC-AUC. These metrics are more meaningful than raw accuracy given the class imbalance identified in the EDA.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))



[[934 102]
 [148 225]]
              precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.69      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.78      0.75      0.76      1409
weighted avg       0.82      0.82      0.82      1409

ROC-AUC: 0.8624879667104869


## 8. Model 2: Random Forest

Random Forest is included as a comparison model. It's a non linear ensemble that tends to perform well on tabular data and picks up interactions between features without needing manual engineering. class_weight= balanced is used to handle the class imbalance directly within the model.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, rf_pred))
print(classification_report(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))


[[943  93]
 [202 171]]
              precision    recall  f1-score   support

           0       0.82      0.91      0.86      1036
           1       0.65      0.46      0.54       373

    accuracy                           0.79      1409
   macro avg       0.74      0.68      0.70      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC: 0.8409703748175598


## 9. Probability Threshold Tuning

By default, a model predicts churn when the probability hits 0.5 or above. That's a reasonable starting point but not necessarily the right one here. Missing a churner costs more than flagging someone who would've stayed, so it's worth testing lower thresholds catching more churners means accepting more false alarms and this section finds where that trade off makes practical sense.
This section tests multiple thresholds to find a better trade off than the default 0.5.


### 9.1 Testing thresholds from 0.30 to 0.50


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.5, 0.45, 0.4, 0.35, 0.3]

for t in thresholds:
    preds = (y_prob >= t).astype(int)
    p = precision_score(y_test, preds)
    r = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    print(f"threshold={t:.2f}  precision={p:.2f}  recall={r:.2f}  f1={f1:.2f}")


threshold=0.50  precision=0.69  recall=0.60  f1=0.64
threshold=0.45  precision=0.64  recall=0.64  f1=0.64
threshold=0.40  precision=0.60  recall=0.70  f1=0.65
threshold=0.35  precision=0.57  recall=0.74  f1=0.64
threshold=0.30  precision=0.55  recall=0.80  f1=0.65


**Finding:** Lowering the threshold below 0.5 substantially improves recall while only modestly reducing precision a favourable trade off for the retention use case. A threshold of around 0.40 offers the best balance based on F1-score.


### 9.2 Final evaluation at threshold 0.40


In [ ]:
t = 0.40
y_pred_040 = (y_prob >= t).astype(int)

from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_test, y_pred_040))
print(classification_report(y_test, y_pred_040))


[[862 174]
 [111 262]]
              precision    recall  f1-score   support

           0       0.89      0.83      0.86      1036
           1       0.60      0.70      0.65       373

    accuracy                           0.80      1409
   macro avg       0.74      0.77      0.75      1409
weighted avg       0.81      0.80      0.80      1409



## 10. Feature Importance Analysis

Examining the Logistic Regression coefficients to identify which features most strongly increase or decrease the predicted probability of churn. This provides interpretability a key requirement for many business stakeholders who need to understand why a model is flagging certain customers.


In [ ]:
import pandas as pd

feature_importance = pd.Series(model.coef_[0], index=X.columns).sort_values()

print("Top features that DECREASE churn:")
print(feature_importance.head(10))

print("\nTop features that INCREASE churn:")
print(feature_importance.tail(10))


Top features that DECREASE churn:
Contract_Two year                      -1.380764
Contract_One year                      -0.629032
OnlineSecurity_Yes                     -0.388737
TechSupport_Yes                        -0.309641
Dependents_Yes                         -0.158767
OnlineBackup_No internet service       -0.158415
InternetService_No                     -0.158415
OnlineSecurity_No internet service     -0.158415
DeviceProtection_No internet service   -0.158415
TechSupport_No internet service        -0.158415
dtype: float64

Top features that INCREASE churn:
DeviceProtection_Yes              0.017902
Partner_Yes                       0.055978
SeniorCitizen                     0.161974
StreamingTV_Yes                   0.288582
MultipleLines_Yes                 0.302472
PaymentMethod_Electronic check    0.315375
PaperlessBilling_Yes              0.335176
MultipleLines_No phone service    0.353731
StreamingMovies_Yes               0.388066
InternetService_Fiber optic       1.050

Two year contract customers are the least likely to churn by a wide margin. One year contracts help too but not nearly as much. Customers with online security or tech support also tend to stay longer.

On the other side fibre optic internet is the biggest churn driver in the entire model. Customers on it are significantly more likely to leave than anyone else. Paperless billing and electronic check payments also increase churn risk but nowhere near as much as fibre optic.

so customers locked into longer contracts with support services stay. Customers on fibre optic, paying manually with no add-ons leave.

## 11. Model Persistence
The trained model and feature list are saved to disk with joblib. This means the model can be loaded and used for predictions later without retraining from scratch.

In [ ]:
import joblib

joblib.dump(model, "logreg_churn_model.pkl")
joblib.dump(X.columns, "model_features.pkl")


['model_features.pkl']

## 12. Conclusion and Reflection

### Summary of Work
A supervised ML pipeline was built to predict customer churn using the Telco dataset. The work covered data cleaning, exploratory analysis, one hot encoding, training two models (Logistic Regression and Random Forest), probability threshold tuning and feature importance analysis.

### Key Outcomes
- Contract type, tenure, payment method and add on services are the strongest predictors of churn.
- Lowering the classification threshold to 0.40 caught more churners than the default 0.5 with only a small drop in precision.
- Logistic Regression coefficients pointed directly to which customer behaviours a retention team should target.

### Limitations
- At roughly 7,000 rows, the dataset is small for a real world deployment and may not capture the full range of churn behaviour.
- All EDA was tabular and charts would make the patterns easier to present to stakeholders.
- More powerful models like XGBoost weren't tested here but would be a logical next step.

### Reflection
This project reinforced for me that strong data science work depends as much on understanding the business context as on technical accuracy. Tuning the probability threshold based on the unequal cost of missing a churner vs flagging someone who would've stayed was a clear example of how a small technical decision can meaningfully change the value of a model in practice.
